# **Generazione dataset sintetici**

**Generazione di dataset sintetici con CT-GAN e T-VAE**

**Installation**

In [7]:
%pip install sdv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [50]:
import sdv
import pandas as pd
import os
#print(sdv.version.public)

In [51]:
step2 = "../../data/Step2/"
if not os.path.exists(step2 + "Output"):
    os.makedirs(step2 + "Output")

!mv ../../data/*.* {step2}Output

mv: cannot stat '../../data/*.*': No such file or directory


**Parameters**

In [52]:
synthesizer_type="TVAE" # da settare
#synthesizer_type="CTGAN" # da settare

R='N' # da settare

if R != "N":
    PR=10 #da settare
else:
    PR=0



In [62]:
input_dir = rf"/home/onyxia/work/data/Step1/Output/"
output_dir = rf"/home/onyxia/work/data/Step2/Output/"

input_file_name = os.path.join(
    input_dir,
    f"real_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2.csv"
)

output_file_name = os.path.join(
    output_dir,
    f"synthetic_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2_{synthesizer_type}.csv"
)

metadata_file_name = os.path.join(
    output_dir,
    f"{synthesizer_type}_metadata_health_M10_TH20_R{R}_PR{PR}_4CAT.json"
)

modello = os.path.join(
    output_dir,
    f"{synthesizer_type}_Modello_Sintetizzatore_M10_TH20_R{R}_PR{PR}_4CAT.pkl"
)



In [63]:
print(input_file_name)

/home/onyxia/work/data/Step1/Output/real_data_datasetM10_TH20_RN_PR0_4CAT_MISS_X2.csv


In [64]:
dataset_health = pd.read_csv(input_file_name, sep=',')

In [66]:
dataset_health.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      10000 non-null  int64  
 1   municipality_residence  10000 non-null  object 
 2   age                     10000 non-null  int64  
 3   civil_status            10000 non-null  object 
 4   gender                  10000 non-null  object 
 5   occupation              10000 non-null  int64  
 6   physical_activity       10000 non-null  object 
 7   genetic_predisposition  10000 non-null  object 
 8   birth_year              10000 non-null  int64  
 9   birth_month             10000 non-null  int64  
 10  birth_day               10000 non-null  int64  
 11  age_norm                10000 non-null  float64
 12  activity_norm           10000 non-null  float64
 13  predisposition_norm     10000 non-null  float64
 14  score1                  10000 non-null 

In [67]:
dataset_health.shape

(10000, 22)

In [68]:
columns_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 'age_norm', 'activity_norm', 'predisposition_norm','birth_year','birth_month','birth_day']
dataset_health = dataset_health.drop(columns=columns_to_drop)

In [69]:
dataset_health.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   municipality_residence  10000 non-null  object
 1   age                     10000 non-null  int64 
 2   civil_status            10000 non-null  object
 3   gender                  10000 non-null  object
 4   occupation              10000 non-null  int64 
 5   physical_activity       10000 non-null  object
 6   genetic_predisposition  10000 non-null  object
 7   diagnosis               10000 non-null  int64 
 8   municipality_birth      10000 non-null  object
 9   birth_dayofyear         10000 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 781.4+ KB


In [70]:
dataset_health.shape

(10000, 10)

In [71]:
from sdv.metadata import Metadata

metadata_health = Metadata.detect_from_dataframe(
    data=dataset_health,
    table_name='health_metadata')

In [72]:
metadata_health

{
    "tables": {
        "health_metadata": {
            "columns": {
                "municipality_residence": {
                    "sdtype": "categorical"
                },
                "age": {
                    "sdtype": "numerical"
                },
                "civil_status": {
                    "sdtype": "categorical"
                },
                "gender": {
                    "sdtype": "categorical"
                },
                "occupation": {
                    "sdtype": "categorical"
                },
                "physical_activity": {
                    "sdtype": "categorical"
                },
                "genetic_predisposition": {
                    "sdtype": "categorical"
                },
                "diagnosis": {
                    "sdtype": "categorical"
                },
                "municipality_birth": {
                    "sdtype": "categorical"
                },
                "birth_dayofyear": {
         

In [73]:
# Se il file json di metadati esiste lo cancello
if os.path.exists(metadata_file_name):
    os.remove(metadata_file_name)

# Salvo (verrà creato un nuovo file)
metadata_health.save_to_json(metadata_file_name)

In [74]:
'''
if synthesizer_type =="CTGAN":
  from sdv.single_table import CTGANSynthesizer

  synthesizer = CTGANSynthesizer(metadata_health)
  synthesizer.fit(dataset_health)


if synthesizer_type =="TVAE":

  from sdv.single_table import TVAESynthesizer

  synthesizer = TVAESynthesizer(metadata_health)
  synthesizer.fit(dataset_health)
'''


'\nif synthesizer_type =="CTGAN":\n  from sdv.single_table import CTGANSynthesizer\n\n  synthesizer = CTGANSynthesizer(metadata_health)\n  synthesizer.fit(dataset_health)\n\n\nif synthesizer_type =="TVAE":\n\n  from sdv.single_table import TVAESynthesizer\n\n  synthesizer = TVAESynthesizer(metadata_health)\n  synthesizer.fit(dataset_health)\n'

In [75]:
####OK TVAE
if synthesizer_type == "CTGAN":
    from sdv.single_table import CTGANSynthesizer

    # Creazione del modello CTGAN con parametri ottimizzati
    synthesizer = CTGANSynthesizer(
        metadata=metadata_health,  # metadata già definito con tipi variabili
        epochs=1000,                # numero di epoche
                                   # Per dataset ~10K: abbastanza alto per far convergere GAN, ma non troppo da overfit
        batch_size=500,            # dimensione del batch
                                   # Batch medio stabilizza il training; batch troppo piccolo = rumore, troppo grande = lenta convergenza
        generator_dim=(256, 256),  # dimensioni rete generatore
                                   # Reti più grandi = più capacità di catturare relazioni complesse tra variabili, ma più parametri = più rischio overfitting
        discriminator_dim=(256, 256),  # dimensioni rete discriminatore
                                       # Simile al generatore, ma discriminatore leggermente potente aiuta a ridurre mode collapse
        generator_lr=1e-4,         # learning rate del generatore
                                   # Piccolo per stabilità, troppo grande = oscillazioni
        discriminator_lr=2e-4,     # learning rate del discriminatore
                                   # Di solito leggermente più alto del generatore per bilanciare training
        discriminator_steps=5,     # quante volte aggiornare il discriminatore per ogni passo del generatore
                                   # Più step = discriminatore più forte, utile quando ci sono molte categorie rare (~10K dataset)
        pac=10,                    # packed discriminator
                                   # Raggruppa 10 record insieme per giudicare “reale/sintetico” → aiuta le categorie rare
        verbose=True               # stampa i progressi del training
    )

    synthesizer.fit(dataset_health)

elif synthesizer_type == "TVAE":
    from sdv.single_table import TVAESynthesizer

    # Creazione del modello TVAE con parametri robusti
    synthesizer = TVAESynthesizer(
        metadata=metadata_health,
        epochs=1000,                # Numero di epoche simile a CTGAN, sufficiente per dataset ~10K
        batch_size=500,            # Batch medio per stabilità training
        embedding_dim=128,         # Dimensione embedding delle categorie
                                   # Più alto = cattura più informazioni, ma più complesso; 128 è un buon compromesso per ~10K record
        compress_dims=(256, 256),  # Dimensioni della rete di compressione (encoder)
                                   # Serve a comprimere le features in embedding latente; dimensioni simili a CTGAN bilanciano capacità e overfitting
        decompress_dims=(256, 256),# Dimensioni della rete di decompressione (decoder)
                                   # Deve avere capacità simile all’encoder per ricostruire bene il dataset
        loss_factor=2,             # Peso della loss discreta vs continua
                                   # Maggiore = attenzione sulle variabili categoriali, utile con molte categorie
        verbose=True               # Stampa i progressi del training
    )

    synthesizer.fit(dataset_health)

else:
    raise ValueError(f"Unsupported synthesizer_type: {synthesizer_type}")

"""
loss_factor
0.1 = Ultra-focus numeriche (code, varianza)
1.0 = Equilibrato
2.0 = Default (categoriche)
3.0+ = Forza categoriche (rischio loss alta)
"""

Loss: +13.49: 100%|██████████| 1000/1000 [2:06:01<00:00,  7.56s/it] 


'\nloss_factor\n0.1 = Ultra-focus numeriche (code, varianza)\n1.0 = Equilibrato\n2.0 = Default (categoriche)\n3.0+ = Forza categoriche (rischio loss alta)\n'

In [76]:
synthetic_data = synthesizer.sample(num_rows= 10000)
synthetic_data.head()

,municipality_residence,age,civil_status,gender,occupation,physical_activity,genetic_predisposition,diagnosis,municipality_birth,birth_dayofyear
0,Taranto,39,Married,Male,1,4,4,1,Torino,331
1,Taranto,52,Married,Female,1,3,6,2,Pavia,341
2,Galatone,56,Married,Male,1,4,6,2,Taranto,243
3,Lecce,35,NeverMarried,Male,1,2,7,4,Matera,256
4,Bagnolo del Salento,22,Married,Male,0,4,4,1,Ravenna,213


In [77]:
print(modello)

/home/onyxia/work/data/Step2/Output/TVAE_Modello_Sintetizzatore_M10_TH20_RN_PR0_4CAT.pkl


In [78]:
synthesizer.save(modello)

In [79]:
print(synthesizer.get_parameters())

{'enforce_min_max_values': True, 'enforce_rounding': True, 'embedding_dim': 128, 'compress_dims': (256, 256), 'decompress_dims': (256, 256), 'l2scale': 1e-05, 'batch_size': 500, 'verbose': True, 'epochs': 1000, 'loss_factor': 2, 'enable_gpu': True}


In [80]:
print(output_file_name)

/home/onyxia/work/data/Step2/Output/synthetic_data_datasetM10_TH20_RN_PR0_4CAT_MISS_X2_TVAE.csv


In [81]:
# Export the dataframe to a CSV file with a comma separator.
synthetic_data.to_csv(output_file_name, sep=',', index=False)